<h1>Chapters 5 & 6 - Skills</h1>
<i>Adding specialized skills to your Agent that are used with ReAct.</i>


<a href="..."><img src="https://img.shields.io/badge/Buy%20the%20Book!-grey?logo=amazon"></a>
<a href="..."><img src="https://img.shields.io/badge/O'Reilly-white.svg?logo=data:image/svg%2bxml;base64,PHN2ZyB3aWR0aD0iMzQiIGhlaWdodD0iMjciIHZpZXdCb3g9IjAgMCAzNCAyNyIgZmlsbD0ibm9uZSIgeG1sbnM9Imh0dHA6Ly93d3cudzMub3JnLzIwMDAvc3ZnIj4KPGNpcmNsZSBjeD0iMTMiIGN5PSIxNCIgcj0iMTEiIHN0cm9rZT0iI0Q0MDEwMSIgc3Ryb2tlLXdpZHRoPSI0Ii8+CjxjaXJjbGUgY3g9IjMwLjUiIGN5PSIzLjUiIHI9IjMuNSIgZmlsbD0iI0Q0MDEwMSIvPgo8L3N2Zz4K"></a>
<a href="..."><img src="https://img.shields.io/badge/GitHub%20Repository-black?logo=github"></a>
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](...)

---

This notebook is for Chapters 5 and 6 of the [An Illustrated Guide to AI Agents](...) book by [Maarten Grootendorst](https://www.linkedin.com/in/mgrootendorst/) and [Jay Alammar](https://www.linkedin.com/in/jalammar).

---

<a href="...">
<img src="https://learning.oreilly.com/covers/urn:orm:book:9798341662681/400w/" width="350"/></a>


### **[OPTIONAL]** - Installing Packages on Google Colab <img src="https://upload.wikimedia.org/wikipedia/commons/d/d0/Google_Colaboratory_SVG_Logo.svg" width=100>

If you are viewing this notebook on Google Colab (or any other cloud vendor), you need to **uncomment and run** one of the following codeblock to install the dependencies for this chapter. If you want to use a cloud provider, you only need to run the following code block:

In [ ]:
# %%capture
# !pip install illustrated-agents

---

💡 **NOTE**: If you want to use the GPU with `ollama`, then you will have to select a GPU first. In Google Colab, go to
**Runtime > Change runtime type > Hardware accelerator > GPU > GPU type > T4**. 

Then, **uncomment** and run this codeblock:

---

In [ ]:
# !apt-get install -y zstd > /dev/null 2>&1 && curl -fsSL https://ollama.com/install.sh | sh
# !nohup ollama serve > /dev/null 2>&1 & sleep 3 && ollama pull gemma4:e4b &

<hr style="height: 5px; border: none; border-radius: 5px; background: linear-gradient(to right, #000000, #7D7D7D);" />

## 1 - Choosing Your LLM

At the beginning of every chapter, we start by choosing the LLM that we want to use. The skills that we are going to explore can be activated as if they are tools. As such, you can either use the non-native (Gemma 3) or native (Gemma 4) since both have tool calling capabilities that work with skills. For simplicity, we are going with Gemma 4 since we continue to focus more on native capabilities from this point onward:

In [1]:
import os
from illustrated_agents.chapters.ch5_native import LLM

# Ollama through OpenAI API
llm = LLM(model="gemma4:e4b", backend="openai", api_base="http://localhost:11434/v1/", think=True)

# Llama.cpp server
# llm = LLM(model="openai/gemma-4-E4B-it-Q4_K_M", backend="litellm", api_base="http://localhost:8080", think=True)

# LM Studio
# llm = LLM(model="lm_studio/gemma-4-E4B-it", backend="litellm", api_base="http://localhost:1234/v1", think=True)

# Google's Gemini / Gemma
# os.environ['GEMINI_API_KEY'] = "YOUR_API_KEY"
# llm = LLM(model="gemini/gemini-2.5-flash", backend="litellm", api_key=None)

## 2 - Adding Recipes with **`SKILL.md`**

With modules like tools and MCP, we can give an Agent access to a number of tools or actions that it can take. However, when exactly to use those actions and how they fit into a larger workflow is not covered by any of these modules. This is where `SKILL.md` come in, they are a set of instructions on how to perform a given task. For instance, if you want it to create a presentation, you might need it to first search the web for relevant information (this is a tool), then summarize this information (it can do this by itself), and then finally create the slides one at a time (this is also a tool). This workflow or "recipe" for creating a presentation might therefore include a set of tools but also instructions on how to use them and in what order. As such,

> Skills teach your Agent what to do, when to do it, and how

The format of a `SKILL.md` file allows for proper context engineering. In the yaml frontmatter, there is the basic description of your skill which is always loaded into the context window:

```yaml
name: ...
description: ...
```

Below that, there is a more extensive description of the skill and how it should be executed. The full structure then becomes something like this:


```markdown
---
name: ...
description ...
---

#
...

##
...
```

This structure is especially helpful as you can write down extensive descriptions on how to use the skill, which may include domain-specific information. Skills are therefore especially helpful when you notice you have to repeat prompts often, like having to explain everytime the tone of voice that you want to or some domain-specific information regarding the schemas of your database.

Another benefit of skills is that they are **progressively disclosed**. This means that the yaml frontmatter is always given to the Agent as a system prompt, much like the tools we constructed in Chapter 5. However, the full markdown description is only given when the skill is **activated**. This therefore occupies a small amount of the context window and extends only when the skill is activated.

Next, let's explore how to give your `TinyAgent` access to these skills by first creating the `Skills` module:

In [2]:
from pathlib import Path

import yaml


class Skills:
    """Skill loader for the Agent.

    Skills are recipes that teach the agent what to do, when to do it, and how.
    They are defined in SKILL.md files with YAML frontmatter (name, description)
    and markdown instructions.

    The skills are loaded progressively. As such, the name and description are
    available in the system prompt, but the full instructions are only injected
    when the agent **activates** a skill.
    """

    def __init__(self):
        self.skills = {}

    def add_skill(self, path: str):
        """Load a skill from a SKILL.md file"""
        content = Path(path).read_text(encoding="utf-8")

        # Split on YAML delimiters and extract the frontmatter and body
        parts = content.split("---", 2)

        # Frontmatter
        frontmatter = yaml.safe_load(parts[1])
        name = frontmatter["name"]
        description = frontmatter["description"]

        # Body
        body = parts[2].strip()

        # Store the skill
        self.skills[name] = {
            "description": description,
            "instructions": body,
        }

    def activate(self, tool_call: dict) -> str:
        """Activate a skill and return formatted observation.

        Arguments:
            tool_call: A parsed tool call dict with "tool" and "kwargs" keys.

        Returns:
            Formatted observation with skill instructions.
        """
        name = tool_call["kwargs"]["name"]
        if name in self.skills:
            instructions = self.skills[name]["instructions"]
            return f"Skill '{name}' activated. Follow these instructions:\n\n{instructions}"
        return f"Skill '{name}' not found. Available skills: {', '.join(self.skills.keys())}"

    @property
    def prompt(self) -> str:
        """Generate prompt with skill descriptions (not full instructions)."""
        return f"""
# Skills

You have specialized skills available. To use a skill,
call it like a tool by referencing their name.

The skill will provide detailed instructions for completing the task.

Available skills:
{self.descriptions}
"""

    @property
    def descriptions(self) -> str:
        """Get short descriptions of all skills."""
        return "\n".join(f"- `{name}`: {skill['description']}" for name, skill in self.skills.items())

    def as_tool(self, name: str):
        """Convert a skill's instruction to a function so it can be called as a tool."""
        def skill_as_a_tool():
            return self.skills[name]["instructions"]
        skill_as_a_tool.__name__ = name
        return skill_as_a_tool

Much like with the `Tools` module, there are only a couple of functions that we really need to add skills, starting with the prompt:

In [3]:
from illustrated_agents.chapters.ch6_skills import skills_load_annotated; skills_load_annotated

In [4]:
from illustrated_agents.chapters.ch6_skills import skills_class_annotated; skills_class_annotated

In [5]:
from illustrated_agents.chapters.ch6_skills import skills_activate_annotated; skills_activate_annotated

In [6]:
from illustrated_agents.chapters.ch6_skills import skills_as_tool_annotated; skills_as_tool_annotated

Now that you have explored the code for loading and activating skills, let's explore how to actually create a skill. We already have prepared a skill for you to use, which can be found in `src/illustrated_agents/skills/file_analyzer`. The file contains all information about how to use our previously defined tool (`read_markdown`) along with a set of instructions on how to summarize its content. Let's load the skill and inspect it:

In [7]:
import illustrated_agents

# We choose the file_analyzer skill as an example
file_analyzer_path = Path(illustrated_agents.__file__).parent / "skills" / "file_analyzer" / "SKILL.md"

# Load the skill
skills = Skills()
skills.add_skill(file_analyzer_path)

Let's inspect the skill's description:

In [8]:
print(skills.skills["file_analyzer"]["description"])

Analyze files and provide structured summaries


This is a short description of what the task is, but how to actually do it is covered in the full instruction:

In [9]:
print(skills.skills["file_analyzer"]["instructions"])

# File Analyzer Skill

When asked to analyze a file or URL, follow these steps:

## Step 1: Read the Content

Use the `read_markdown` tool to fetch the file content from the provided path or URL.

## Step 2: Analyze the Structure

Identify:
- The type of document (README, documentation, code, article, etc.)
- Main sections and their purposes
- Key topics covered

## Step 3: Provide Structured Output

Always format your analysis as:

**Document Type**: [type of document]

**Purpose**: [one sentence describing the main purpose]

**Sections**:
- [Section 1]: [brief description]
- [Section 2]: [brief description]
- ...

**Key Points**:
- [Important point 1]
- [Important point 2]
- [Important point 3]

**Summary**: [2-3 sentence summary of the entire document]

## Guidelines

- Be concise but comprehensive
- Focus on the most important information
- If the document is code-related, mention technologies and dependencies
- If it's a README, highlight installation and usage instructions


Activating the skill would give back the full instruction as an observation:

In [10]:
print(skills.activate({"tool": "use_skill", "kwargs": {"name": "file_analyzer"}}))

Skill 'file_analyzer' activated. Follow these instructions:

# File Analyzer Skill

When asked to analyze a file or URL, follow these steps:

## Step 1: Read the Content

Use the `read_markdown` tool to fetch the file content from the provided path or URL.

## Step 2: Analyze the Structure

Identify:
- The type of document (README, documentation, code, article, etc.)
- Main sections and their purposes
- Key topics covered

## Step 3: Provide Structured Output

Always format your analysis as:

**Document Type**: [type of document]

**Purpose**: [one sentence describing the main purpose]

**Sections**:
- [Section 1]: [brief description]
- [Section 2]: [brief description]
- ...

**Key Points**:
- [Important point 1]
- [Important point 2]
- [Important point 3]

**Summary**: [2-3 sentence summary of the entire document]

## Guidelines

- Be concise but comprehensive
- Focus on the most important information
- If the document is code-related, mention technologies and dependencies
- If it's a

As you can see, the instruction is quite long and adding that to the system prompt would quickly fill up the context window if you have several skills that your Agent can use. 

Note that we also added the `as_tool` function which allows us to convert the main description to a function. When you run the function, it does nothing more than give back the full description of the skill.

In [11]:
file_analyzer = skills.as_tool("file_analyzer")
file_analyzer()

"# File Analyzer Skill\n\nWhen asked to analyze a file or URL, follow these steps:\n\n## Step 1: Read the Content\n\nUse the `read_markdown` tool to fetch the file content from the provided path or URL.\n\n## Step 2: Analyze the Structure\n\nIdentify:\n- The type of document (README, documentation, code, article, etc.)\n- Main sections and their purposes\n- Key topics covered\n\n## Step 3: Provide Structured Output\n\nAlways format your analysis as:\n\n**Document Type**: [type of document]\n\n**Purpose**: [one sentence describing the main purpose]\n\n**Sections**:\n- [Section 1]: [brief description]\n- [Section 2]: [brief description]\n- ...\n\n**Key Points**:\n- [Important point 1]\n- [Important point 2]\n- [Important point 3]\n\n**Summary**: [2-3 sentence summary of the entire document]\n\n## Guidelines\n\n- Be concise but comprehensive\n- Focus on the most important information\n- If the document is code-related, mention technologies and dependencies\n- If it's a README, highlight i

## 3 - Updating your **`TinyAgent`**

Now, to implement the skill into your `TinyAgent`, we follow the same structure as we would with the tool usage since we are going to activate any given skill as if it were a tool. The changes to your `TinyAgent` are as follows:

In [12]:
from illustrated_agents.chapters.ch2 import Response
from illustrated_agents.chapters.ch5_native import LLM, Memory
from illustrated_agents.chapters.ch6 import ReAct, Tools


class TinyAgent:
    """A minimal, modular, and educational agent framework."""

    def __init__(self, llm: LLM, memory: Memory, tools: Tools, planner: ReAct, skills: Skills):
        self.llm = llm
        self.memory = memory
        self.tools = tools
        self.planner = planner
        self.skills = skills

        # Build system prompt with all components
        system_prompt = "You are a helpful assistant.\n"
        system_prompt += self.planner.prompt + "\n"
        system_prompt += self.tools.prompt + "\n"
        system_prompt += self.skills.prompt + "\n"
        self.memory.add("system", system_prompt)

    def run(self, task: str) -> str:
        """Run the agent on a task."""
        self.memory.add("user", task)

        # `Autonomy` loop
        for step in range(self.planner.max_steps):
            result = self._step()
            if result is not None:
                return result

        return "Max steps reached without completion."

    def _step(self) -> str:
        """Perform a single step."""
        # THOUGHT: Generate response and add to memory
        response = self.llm.generate(self.memory.get_messages(), tools=self.tools.schemas)
        self.memory.add("assistant", response.content, tool_call=response.tool_call)

        # Tool parsing
        response = self.tools.parse(response)

        # ANSWER: Stopping mechanism
        if self.tools.is_done(response):
            return response.content

        return self._execute_action(response)

    def _execute_action(self, response: Response) -> None:
        """Execute a tool action."""

        # ACTION: execute tools
        result = self.tools.execute(response)

        # OBSERVATION: add tool results to memory and display
        role, observation = self.tools.observation(result)
        self.memory.add(role, observation)

        return None

These changes result in your new `TinyAgent`:

In [13]:
from illustrated_agents.chapters.ch6_skills import tinyagents_diff; tinyagents_diff

Finally, let's see if your newly updated `TinyAgent` will properly summarize any given markdown by using the skill and not only the tool:

In [14]:
import urllib.request
from illustrated_agents.chapters.ch2 import Response
from illustrated_agents.chapters.ch5_native import NativeTools, LLM, Memory
from illustrated_agents.chapters.ch6_native import NativeReAct

def read_markdown(url: str) -> str:
    """Read a markdown file from a given `url`"""
    return urllib.request.urlopen(url).read().decode()

# Memory
memory = Memory()

# ReAct
react = NativeReAct(max_steps=10)

# Skills
skills = Skills()
skills.add_skill(file_analyzer_path)

# Tools
tools = NativeTools()
tools.add_tool("read_markdown", read_markdown)
tools.add_tool("file_analyzer", file_analyzer)

# Create agent
agent = TinyAgent(
    llm=llm, 
    tools=tools, 
    memory=memory, 
    planner=react, 
    skills=skills
)

In [15]:
query = "Use the `file_analyzer` skill."

print(agent.run(query))

I have analyzed the `file_analyzer` skill documentation.

This skill is designed to analyze the content of a file or a URL and provide a structured summary, including the document type, purpose, sections, key points, and a summary.

**To use this skill, please provide me with:**

1.  **A file path** (if the file is accessible in our current context).
2.  **A URL** (so I can use the `read_markdown` tool to fetch the content).

Once you provide the content source, I will follow these steps to give you a detailed analysis:

*   **Document Type**: Identify what kind of document it is.
*   **Purpose**: State its main goal in one sentence.
*   **Sections**: Outline the main topics covered.
*   **Key Points**: List the most important takeaways.
*   **Summary**: Give a 2-3 sentence overall summary.

**What would you like me to analyze?**


In [16]:
from rich import print as pprint

pprint(agent.memory.get_messages())

[
    {
        'role': 'system',
        'content': 'You are a helpful assistant.\n\n\n\n# Skills\n\nYou have specialized skills available. To use a
skill,\ncall it like a tool by referencing their name.\n\nThe skill will provide detailed instructions for 
completing the task.\n\nAvailable skills:\n- `file_analyzer`: Analyze files and provide structured summaries\n\n'
    },
    {'role': 'user', 'content': 'Use the `file_analyzer` skill.'},
    {
        'role': 'assistant',
        'content': '',
        'tool_calls': [
            {
                'id': 'call_m7leynfb',
                'function': {'arguments': '{}', 'name': 'file_analyzer'},
                'type': 'function',
                'index': 0
            }
        ]
    },
    {
        'role': 'tool',
        'content': "# File Analyzer Skill\n\nWhen asked to analyze a file or URL, follow these steps:\n\n## Step 1:
Read the Content\n\nUse the `read_markdown` tool to fetch the file content from the provided path or URL.\n\n## 
Step 2: Analyze the Structure\n\nIdentify:\n- The type of document (README, documentation, code, article, etc.)\n- 
Main sections and their purposes\n- Key topics covered\n\n## Step 3: Provide Structured Output\n\nAlways format 
your analysis as:\n\n**Document Type**: [type of document]\n\n**Purpose**: [one sentence describing the main 
purpose]\n\n**Sections**:\n- [Section 1]: [brief description]\n- [Section 2]: [brief description]\n- ...\n\n**Key 
Points**:\n- [Important point 1]\n- [Important point 2]\n- [Important point 3]\n\n**Summary**: [2-3 sentence 
summary of the entire document]\n\n## Guidelines\n\n- Be concise but comprehensive\n- Focus on the most important 
information\n- If the document is code-related, mention technologies and dependencies\n- If it's a README, 
highlight installation and usage instructions"
    },
    {
        'role': 'assistant',
        'content': 'I have analyzed the `file_analyzer` skill documentation.\n\nThis skill is designed to analyze 
the content of a file or a URL and provide a structured summary, including the document type, purpose, sections, 
key points, and a summary.\n\n**To use this skill, please provide me with:**\n\n1.  **A file path** (if the file is
accessible in our current context).\n2.  **A URL** (so I can use the `read_markdown` tool to fetch the 
content).\n\nOnce you provide the content source, I will follow these steps to give you a detailed analysis:\n\n*  
**Document Type**: Identify what kind of document it is.\n*   **Purpose**: State its main goal in one sentence.\n* 
**Sections**: Outline the main topics covered.\n*   **Key Points**: List the most important takeaways.\n*   
**Summary**: Give a 2-3 sentence overall summary.\n\n**What would you like me to analyze?**'
    }
]

Now let's try to see if the model can use it without being instructed to do so, we'll first need to clear it's memory and then run a more complex query:

In [17]:
# Create a new agent
agent = TinyAgent(
    llm=llm, 
    tools=tools, 
    memory=memory, 
    planner=react, 
    skills=skills
)

# Run a more complex query
query = "Analyze the file at https://raw.githubusercontent.com/MaartenGr/BERTopic/refs/heads/master/README.md."
print(agent.run(query))

**Document Type**: Software Library Documentation (README)

**Purpose**: To provide a comprehensive guide on the BERTopic library, detailing its architecture, installation, usage, and advanced features for state-of-the-art topic modeling.

**Sections**:
*   **Introduction & Overview**: Highlights that BERTopic combines 🤗 transformers with c-TF-IDF to create easily interpretable, dense topic clusters.
*   **Core Modalities**: Lists support for various topic modeling scenarios, including Guided, Supervised, Semi-supervised, Hierarchical, Multimodal, and LLM-based topics.
*   **Installation**: Provides instructions for installation using `uv` or `pip`, with options for installing with specific embedding backends (e.g., `[vision]`).
*   **Quick Start**: Offers a basic "Hello World" example using the `fetch_20newsgroups` dataset to demonstrate fitting and retrieving topic information (`get_topic_info`, `get_topic`).
*   **Advanced Features**: Covers techniques like fine-tuning topic represe

The output is exactly as instructed by our particular skill, great! To check how it used the skill, let's see how it did by exploring it's memory:

In [18]:
pprint(agent.memory.get_messages())

[
    {
        'role': 'system',
        'content': 'You are a helpful assistant.\n\n\n\n# Skills\n\nYou have specialized skills available. To use a
skill,\ncall it like a tool by referencing their name.\n\nThe skill will provide detailed instructions for 
completing the task.\n\nAvailable skills:\n- `file_analyzer`: Analyze files and provide structured summaries\n\n'
    },
    {'role': 'user', 'content': 'Use the `file_analyzer` skill.'},
    {
        'role': 'assistant',
        'content': '',
        'tool_calls': [
            {
                'id': 'call_m7leynfb',
                'function': {'arguments': '{}', 'name': 'file_analyzer'},
                'type': 'function',
                'index': 0
            }
        ]
    },
    {
        'role': 'tool',
        'content': "# File Analyzer Skill\n\nWhen asked to analyze a file or URL, follow these steps:\n\n## Step 1:
Read the Content\n\nUse the `read_markdown` tool to fetch the file content from the provided path or URL.\n\n## 
Step 2: Analyze the Structure\n\nIdentify:\n- The type of document (README, documentation, code, article, etc.)\n- 
Main sections and their purposes\n- Key topics covered\n\n## Step 3: Provide Structured Output\n\nAlways format 
your analysis as:\n\n**Document Type**: [type of document]\n\n**Purpose**: [one sentence describing the main 
purpose]\n\n**Sections**:\n- [Section 1]: [brief description]\n- [Section 2]: [brief description]\n- ...\n\n**Key 
Points**:\n- [Important point 1]\n- [Important point 2]\n- [Important point 3]\n\n**Summary**: [2-3 sentence 
summary of the entire document]\n\n## Guidelines\n\n- Be concise but comprehensive\n- Focus on the most important 
information\n- If the document is code-related, mention technologies and dependencies\n- If it's a README, 
highlight installation and usage instructions"
    },
    {
        'role': 'assistant',
        'content': 'I have analyzed the `file_analyzer` skill documentation.\n\nThis skill is designed to analyze 
the content of a file or a URL and provide a structured summary, including the document type, purpose, sections, 
key points, and a summary.\n\n**To use this skill, please provide me with:**\n\n1.  **A file path** (if the file is
accessible in our current context).\n2.  **A URL** (so I can use the `read_markdown` tool to fetch the 
content).\n\nOnce you provide the content source, I will follow these steps to give you a detailed analysis:\n\n*  
**Document Type**: Identify what kind of document it is.\n*   **Purpose**: State its main goal in one sentence.\n* 
**Sections**: Outline the main topics covered.\n*   **Key Points**: List the most important takeaways.\n*   
**Summary**: Give a 2-3 sentence overall summary.\n\n**What would you like me to analyze?**'
    },
    {
        'role': 'system',
        'content': 'You are a helpful assistant.\n\n\n\n# Skills\n\nYou have specialized skills available. To use a
skill,\ncall it like a tool by referencing their name.\n\nThe skill will provide detailed instructions for 
completing the task.\n\nAvailable skills:\n- `file_analyzer`: Analyze files and provide structured summaries\n\n'
    },
    {
        'role': 'user',
        'content': 'Analyze the file at 
https://raw.githubusercontent.com/MaartenGr/BERTopic/refs/heads/master/README.md.'
    },
    {
        'role': 'assistant',
        'content': '',
        'tool_calls': [
            {
                'id': 'call_7k94d82j',
                'function': {
                    'arguments': 
'{"url":"https://raw.githubusercontent.com/MaartenGr/BERTopic/refs/heads/master/README.md"}',
                    'name': 'read_markdown'
                },
                'type': 'function',
                'index': 0
            }
        ]
    },
    {
        'role': 'tool',
        'content': '[![PyPI 
Downloads](https://static.pepy.tech/badge/bertopic)](https://pepy.tech/projects/bertopic)\n[![PyPI - 
Python](https://img.shields.io/badge/python-v3.10+-blue.svg)](https://pypi.or

<hr style="height: 5px; border: none; border-radius: 5px; background: linear-gradient(to right, #000000, #7D7D7D);" />

# What We Built

In this chapter, we covered how to give your `TinyAgent` skills which allows for it to get additional context and proper instructions on how to approach certain procedures. 

In [19]:
from illustrated_agents.chapters.ch6_skills import what_we_built; what_we_built

╭───────────────────────────────────────────────── What We Built ─────────────────────────────────────────────────╮
│ TinyAgent                                                                                                       │
│ ├── agent.py    ← Updated (Added instructions in the system prompt for skill activation.)                       │
│ ├── llm.py                                                                                                      │
│ ├── memory.py                                                                                                   │
│ ├── planning.py                                                                                                 │
│ ├── skills.py   ← New (Added `Skills` class for loading and activating skills as tools.)                        │
│ ├── toolbox.py                                                                                                  │
│ └── tools.py                                                                                                    │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

# What's Next

In the next notebook `chapter06_reflection.ipynb` we add a functionality to your `TinyAgent` that allows it to reflect on its behavior. The method is quite straightforward and requires minimal changes!